# XGBoost极值选股器研究流程

使用QuickAdapterV5Dataset和XGBoostExtremaModel进行股票极值预测

In [ ]:
# 过滤警告
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# 加载模块
import polars as pl
import numpy as np

from vnpy.trader.constant import Interval
from vnpy.alpha import AlphaLab, Segment
from vnpy.alpha.dataset.datasets.quick_adapter_v5 import QuickAdapterV5Dataset
from vnpy.alpha.model.models.xgb_extrema_model import XGBoostExtremaModel

# 准备数据

In [ ]:
# 创建数据中心
lab: AlphaLab = AlphaLab("./lab/csi300")

In [ ]:
# 设置任务参数
name = "300_xgb_extrema"
index_symbol: str = "000300.SSE"
start: str = "2010-01-01"
end: str = "2023-12-31"
interval: Interval = Interval.DAILY
extended_days: int = 100

In [ ]:
# 加载成分股代码
component_symbols: list[str] = lab.load_component_symbols(index_symbol, start, end)
print(f"成分股数量: {len(component_symbols)}")

In [ ]:
# 加载成分股数据
df: pl.DataFrame = lab.load_bar_df(component_symbols, interval, start, end, extended_days)
print(f"数据形状: {df.shape}")
df.head(5)

# 创建数据集 (QuickAdapterV5特征 + 极值标签)

In [ ]:
# 创建数据集对象
# freqtrade风格: 使用QuickAdapterV5特征 + argrelextrema极值标签
dataset: QuickAdapterV5Dataset = QuickAdapterV5Dataset(
    df,
    train_period = ("2010-01-01", "2017-12-31"),
    valid_period = ("2018-01-01", "2019-12-31"),
    test_period = ("2020-01-01", "2023-12-31"),
    periods=[10, 20, 30, 40],  # freqtrade特征周期
    label_period_candles=10,  # 极值检测周期
)

In [ ]:
# 收集指数成分过滤器
filters: dict[str, list[str]] = lab.load_component_filters(index_symbol, start, end)

In [ ]:
# 准备特征和标签数据
# 注意: QuickAdapterV5Dataset会自动计算极值标签(&s-extrema)
dataset.prepare_data(filters, max_workers=3)
print(f"特征数量: {len(dataset.feature_results)}")

In [ ]:
# 数据预处理
dataset.process_data()

In [ ]:
# 查看数据集结构
train_df = dataset.fetch_learn(Segment.TRAIN)
print(f"训练数据形状: {train_df.shape}")
print(f"列名: {train_df.columns[:10]}... (共{len(train_df.columns)}列)")
# 检查&开头的列(label列)
label_cols = [c for c in train_df.columns if c.startswith('&')]
print(f"Label列: {label_cols}")

In [ ]:
# 保存数据集
lab.save_dataset(name, dataset)

# 模型训练

In [ ]:
# 从缓存加载数据集
dataset: QuickAdapterV5Dataset = lab.load_dataset(name)

In [ ]:
# 创建XGBoost极值选股模型
# freqtrade风格参数
model: XGBoostExtremaModel = XGBoostExtremaModel(
    learning_rate=0.1,
    max_depth=6,
    n_estimators=100,
    early_stopping_rounds=50,
    num_candles=200,           # 渐进式阈值预热目标
    label_period_candles=10,   # 频率计算周期
)

In [ ]:
# 训练模型
# 注意: 模型会自动识别&s-extrema列作为训练目标
model.fit(dataset)

In [ ]:
# 查看模型细节
model.detail()

In [ ]:
# 保存模型
lab.save_model(name, model)

# 预测信号

In [ ]:
# 加载模型
model: XGBoostExtremaModel = lab.load_model(name)

In [ ]:
# 在测试集上预测
predictions: np.ndarray = model.predict(dataset, Segment.TEST)
print(f"预测数量: {len(predictions)}")
print(f"预测值范围: [{predictions.min():.3f}, {predictions.max():.3f}]")

In [ ]:
# 获取完整结果DataFrame (freqtrade风格)
# 包含: 预测值、阈值、DI值、Weibull参数等
result_df = model.get_result_df()
print(f"结果列: {result_df.columns}")
result_df.head(10)

In [ ]:
"# 查看渐进式阈值\nprint(f\"\\n阈值统计:\")\nprint(f\"Maxima阈值: {result_df['&s-maxima_sort_threshold'].mean():.3f}\")\nprint(f\"Minima阈值: {result_df['&s-minima_sort_threshold'].mean():.3f}\")\nprint(f\"DI_cutoff: {result_df['DI_cutoff'].mean():.3f}\")\nprint(f\"DI_value_mean: {result_df['DI_value_mean'].mean():.3f}\")\nprint(f\"DI_value_std: {result_df['DI_value_std'].mean():.3f}\")"

In [ ]:
"# 根据阈值筛选极值信号 (freqtrade风格)\n# 筛选maxima信号 (预测值 > maxima阈值 → 可能是高点，卖出信号)\nmaxima_signals = result_df.filter(\n    pl.col(\"&s-extrema\") > pl.col(\"&s-maxima_sort_threshold\")\n).select([\"datetime\", \"vt_symbol\", \"&s-extrema\", \"&s-maxima_sort_threshold\"])\nprint(f\"\\nMaxima信号数量: {len(maxima_signals)}\")\nmaxima_signals.head(5)"

In [ ]:
# 筛选minima信号 (预测值 < minima阈值 → 可能是低点)
minima_signals = result_df.filter(
    pl.col("&s-extrema") < pl.col("&s-minima_sort_threshold")
).select(["datetime", "vt_symbol", "&s-extrema", "&s-minima_sort_threshold"])
print(f"\nMinima信号数量: {len(minima_signals)}")
minima_signals.head(5)

In [ ]:
# 构建交易信号
# maxima → 卖出信号 (预测即将到达高点)
# minima → 买入信号 (预测即将到达低点)
buy_signal = minima_signals.with_columns(
    pl.lit(1).alias("signal")
).select(["datetime", "vt_symbol", "signal"])

sell_signal = maxima_signals.with_columns(
    pl.lit(-1).alias("signal")
).select(["datetime", "vt_symbol", "signal"])

# 合并信号
signal_df = pl.concat([buy_signal, sell_signal]).sort(["datetime", "vt_symbol"])
print(f"总信号数量: {len(signal_df)}")
signal_df.head(10)

In [ ]:
# 保存信号数据
lab.save_signal(name, signal_df)

# 策略回测

基于极值预测信号进行回测

In [ ]:
# 加载模块
import importlib
from datetime import datetime
from vnpy.alpha.strategy import BacktestingEngine
import vnpy.alpha.strategy.strategies.equity_demo_strategy as equity_demo_strategy

In [ ]:
# 重载策略类
importlib.reload(equity_demo_strategy)
EquityDemoStrategy = equity_demo_strategy.EquityDemoStrategy

In [ ]:
# 加载信号数据
signal_df = lab.load_signal(name)

In [ ]:
# 创建回测引擎
engine = BacktestingEngine(lab)

# 设置回测参数
engine.set_parameters(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=datetime(2020, 1, 1),
    end=datetime(2023, 12, 31),
    capital=100000000
)

# 添加策略
setting = {"top_k": 20, "n_drop": 3, "hold_thresh": 5}
engine.add_strategy(EquityDemoStrategy, setting, signal_df)

In [ ]:
# 执行回测
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

In [ ]:
# 显示超额收益
engine.show_performance(benchmark_symbol=index_symbol)